## Lab 1 - Sampling

In [19]:
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML

from diffusers import UNet2DModel

import matplotlib.pyplot as plt

from diffusion_utilities import plot_sample

In [12]:
torch.cuda.is_available()

True

### hyperparams

In [13]:
timesteps = 500
beta1 = 1e-4
beta2 = 0.02

# network hyperparameters:
device = "cuda" if torch.cuda.is_available() else "cpu"
n_features = 64
img_dim = 16  # 16x16 image size
n_channels = 3
n_context = 4

save_dir = "./weights"

In [14]:
device

'cuda'

In [15]:
# construct ddpm noise schedule

b_t = (beta2 - beta1) * torch.linspace(0, 1, timesteps + 1, device=device) / timesteps + beta1
a_t = 1 - b_t
ab_t = torch.cumsum(a_t.log(), dim=0).exp()
ab_t[0] = 1.0

In [16]:
def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)
    noise = b_t.sqrt()[t] * z
    mean = (x - pred_noise * ((1. - a_t[t]) / (1 - ab_t[t]).sqrt())) / a_t[t].sqrt()
    return mean + noise 

In [35]:
@torch.no_grad()
def sample_ddpm(model, n_sample, save_rate = 20):
    model.eval()
    samples = torch.randn(n_sample, n_channels, img_dim, img_dim, device=device)
    
    intermediate = []
    for t in tqdm(range(timesteps, 0, -1)):
        time = torch.tensor([t / timesteps]).to(device)

        # sampled noise at this step, only used for t > 1
        z = torch.randn_like(samples) if t > 1 else None

        eps = model(samples, time) # model predicted noise

        samples = denoise_add_noise(samples, t, eps.sample, z)
        if t % save_rate == 0 or t == timesteps or t < 8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate


In [36]:
model = UNet2DModel(
    sample_size=img_dim,  # the target image resolution
    in_channels=n_channels,  # the number of input channels, 3 for RGB images
    out_channels=n_channels,  # the number of output channels
    layers_per_block=2,  # how many layers to use per UNet block
    block_out_channels=(n_features, n_features * 2, n_features * 4),  # More channels -> more capacity
    downsample_padding=1,
    down_block_types=(
        "DownBlock2D",  # a normal downsampling block
        "AttnDownBlock2D",  
        "AttnDownBlock2D", 
    ),
    up_block_types=(
        "AttnUpBlock2D", 
        "AttnUpBlock2D", 
        "UpBlock2D",  # a normal upsampling block
    ),
).to(device)

In [37]:
plt.clf()
samples, intermediate_ddpm = sample_ddpm(model, 32)
animation_ddpm = plot_sample(intermediate_ddpm, 32, 4, save_dir, "ani_run", None, save=False)
HTML(animation_ddpm.to_jshtml())


  0%|          | 0/500 [00:00<?, ?it/s]

<Figure size 640x480 with 0 Axes>